# OCR 실습 노트북 — 전처리부터 Key-Value 추출까지

강의 자료 6.1 ~ 6.8을 **실제로 실행되는 형태**로 정리한 실습 노트북이다.

## 이 노트북이 원본과 다른 점

원본 코드를 그대로 옮기지 않고, 돌려보면서 발견한 문제를 고쳐 넣었다.

| 항목 | 원본 | 이 노트북 |
|---|---|---|
| 금액 정규식 | `합계금액`을 못 잡음 (항상 `None`) | 긴 패턴 우선 + 공백 허용 |
| 날짜 정규식 | `\b\d{4}-./-./\b` — 문법이 깨져 아무것도 못 잡음 | `[-/.]` 구분자 지원 |
| 번호판 문자 | `[가-나\|다-라\|…]` — 파이프가 리터럴, 범위가 1177자 | 실제 40자 나열 |
| Deskew 전경 검출 | `gray < 255` — 실사진에서 조용히 실패 | Otsu 자동 임계값 |
| 한글 폰트 | Windows 경로 하드코딩 | OS 자동 탐색 + 렌더 검증 |
| 명함 스크립트 | 이미지 생성 코드와 OCR 클래스가 한 셀에 섞임 | 분리 |

## 실행 순서

위에서부터 차례대로 실행하면 된다. **1~5번 셀은 필수**고, 그 뒤는 원하는 절만 골라 실행해도 된다.

---
# 1. 환경 점검

먼저 뭐가 깔려 있는지 확인한다. 없는 건 아래 안내대로 설치하면 된다.

In [ ]:
import sys, importlib, platform

PKGS = {
    "numpy":      "numpy",
    "cv2":        "opencv-python",
    "PIL":        "pillow",
    "matplotlib": "matplotlib",
    "easyocr":    "easyocr",
    "pytesseract":"pytesseract",
    "requests":   "requests",
}

print(f"Python {sys.version.split()[0]}  |  {platform.system()} {platform.machine()}")
print("-" * 58)

missing = []
for mod, pkg in PKGS.items():
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, "__version__", "")
        print(f"  [OK]  {pkg:16} {ver}")
    except ImportError:
        print(f"  [--]  {pkg:16} 없음")
        missing.append(pkg)

print("-" * 58)
if missing:
    print("설치 명령 (uv 사용 시):")
    print(f"    uv add {' '.join(missing)}")
    print("설치 명령 (pip 사용 시):")
    print(f"    pip install {' '.join(missing)}")
else:
    print("필요한 패키지가 전부 있다.")

HAS_EASYOCR = "easyocr" not in missing
HAS_TESS_PY = "pytesseract" not in missing

### Tesseract는 파이썬 패키지만으로 안 된다

`pytesseract`는 **연결용 래퍼**일 뿐이라 OS에 엔진 프로그램을 따로 깔아야 한다.

| OS | 설치 |
|---|---|
| **macOS** | `brew install tesseract` + `brew install tesseract-lang` (한국어 포함) |
| **Windows** | [UB-Mannheim 배포판](https://github.com/UB-Mannheim/tesseract/wiki) 설치 시 **Additional language data → Korean** 체크 |
| **Ubuntu** | `sudo apt install tesseract-ocr tesseract-ocr-kor` |

아래 셀이 엔진을 찾아준다.

In [ ]:
import shutil, os

TESSERACT_CMD = None
CANDIDATES = [
    "/opt/homebrew/bin/tesseract",                       # macOS Apple Silicon
    "/usr/local/bin/tesseract",                          # macOS Intel
    "/usr/bin/tesseract",                                # Linux
    r"C:\Program Files\Tesseract-OCR\tesseract.exe",    # Windows
]

found = shutil.which("tesseract")
if found:
    TESSERACT_CMD = found
else:
    for p in CANDIDATES:
        if os.path.exists(p):
            TESSERACT_CMD = p
            break

if TESSERACT_CMD:
    print(f"Tesseract 엔진: {TESSERACT_CMD}")
    if HAS_TESS_PY:
        import pytesseract
        pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD
        try:
            langs = pytesseract.get_languages()
            print(f"설치된 언어: {sorted(langs)}")
            print("한국어(kor) 사용 가능" if "kor" in langs
                  else "한국어(kor)가 없다 → 언어팩을 설치해야 한다")
        except Exception as e:
            print(f"언어 목록 조회 실패: {e}")
else:
    print("Tesseract 엔진을 못 찾았다. 위 표대로 설치하면 된다.")
    print("(Tesseract 없이도 EasyOCR 부분은 전부 실행된다)")

HAS_TESSERACT = TESSERACT_CMD is not None and HAS_TESS_PY

---
# 2. 한글 폰트 자동 탐색

실습에서 가장 자주 막히는 지점이다.

- **`cv2.putText`는 한글을 아예 못 그린다.** 전부 `???`로 나온다 → Pillow를 써야 한다
- Pillow도 폰트 경로를 못 찾으면 `ImageFont.load_default()`로 떨어지는데, **이 기본 폰트 역시 한글을 못 그린다**
- 그래서 "폰트를 찾았다"가 아니라 **"한글이 실제로 그려졌다"**를 확인해야 한다

아래 셀은 OS별 후보를 훑고, **실제로 픽셀을 찍어서** 렌더링을 검증한다.

In [ ]:
import glob
import numpy as np
from PIL import Image, ImageDraw, ImageFont

FONT_CANDIDATES = [
    # (일반, 굵게)
    ("malgun.ttf", "malgunbd.ttf"),                                   # Windows (PATH)
    (r"C:\Windows\Fonts\malgun.ttf", r"C:\Windows\Fonts\malgunbd.ttf"),
    ("/System/Library/Fonts/Supplemental/AppleGothic.ttf",) * 2,      # macOS
    ("/System/Library/Fonts/AppleSDGothicNeo.ttc",) * 2,              # macOS
    ("/Library/Fonts/NanumGothic.ttf", "/Library/Fonts/NanumGothicBold.ttf"),
    ("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",        # Linux
     "/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc"),
    ("/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
     "/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf"),
]


def find_korean_font():
    """OS별 후보를 순회하고, 없으면 시스템 폰트 디렉터리를 훑는다."""
    for reg, bold in FONT_CANDIDATES:
        try:
            ImageFont.truetype(reg, 14)
            return reg, bold
        except OSError:
            continue
    # 마지막 수단: 이름에 cjk/nanum/gothic이 들어간 폰트를 찾는다
    pats = ["/usr/share/fonts/**/*.tt[cf]", "/Library/Fonts/**/*.tt[cf]",
            os.path.expanduser("~/Library/Fonts/**/*.tt[cf]")]
    for pat in pats:
        for p in glob.glob(pat, recursive=True):
            if any(k in p.lower() for k in ("cjk", "nanum", "gothic", "malgun")):
                return p, p
    return None, None


FONT_REG, FONT_BOLD = find_korean_font()


def F(size, bold=False):
    """폰트 객체를 돌려준다. 실패하면 기본 폰트(한글 불가)."""
    path = FONT_BOLD if bold else FONT_REG
    if path is None:
        return ImageFont.load_default()
    try:
        return ImageFont.truetype(path, size)
    except OSError:
        return ImageFont.load_default()


def can_render_korean():
    """실제로 '한글' 두 글자를 그려보고 검은 픽셀이 찍히는지 센다."""
    img = Image.new("L", (200, 44), 255)
    ImageDraw.Draw(img).text((5, 5), "한글", fill=0, font=F(26))
    return int((np.array(img) < 128).sum()) > 50


print(f"찾은 폰트: {FONT_REG}")
if can_render_korean():
    print("한글 렌더링 정상")
else:
    print("한글이 그려지지 않는다. 아래 중 하나로 폰트를 설치할 것:")
    print("   macOS  : brew install --cask font-nanum-gothic")
    print("   Ubuntu : sudo apt install fonts-nanum")
    print("   Windows: 맑은 고딕이 기본 설치돼 있어야 정상")

---
# 3. 공통 헬퍼

노트북에서 이미지를 바로 보기 위한 표시 함수와, 회전을 주는 함수다.

In [ ]:
import cv2
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110


def show(*images, titles=None, cols=None, height=4):
    """BGR / 그레이 이미지를 노트북에 나란히 표시한다."""
    n = len(images)
    cols = cols or n
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.2, rows * height))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        if img.ndim == 3:
            axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            axes[i].imshow(img, cmap="gray", vmin=0, vmax=255)
        axes[i].set_title(titles[i] if titles else f"[{i}]", fontsize=10)
        axes[i].axis("off")
    for j in range(n, len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    plt.show()


def rotate_pil(pil_img, deg):
    """PIL 이미지를 OpenCV로 넘겨 지정 각도만큼 기울인다 (실습용 왜곡 주입)."""
    cv_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    h, w = cv_img.shape[:2]
    M = cv2.getRotationMatrix2D((w // 2, h // 2), deg, 1.0)
    return cv2.warpAffine(cv_img, M, (w, h), flags=cv2.INTER_CUBIC,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=(255, 255, 255))


print("헬퍼 준비 완료")

---
# 4. 샘플 이미지 4종 생성

실습에 쓸 이미지를 직접 만든다. 영수증·명함·번호판에는 **일부러 기울기를 넣어서** 뒤에 나올 Deskew가 실제로 일하는지 볼 수 있게 했다.

| 파일 | 문서 | 기울기 | 뽑을 정보 |
|---|---|---|---|
| `business_registration.jpg` | 사업자등록증 | 0° | 사업자번호, 전화번호 |
| `receipt_sample.jpg` | 영수증 | −2.5° | 사업자번호, 전화번호, 거래일자, 금액 |
| `business_card_sample.jpg` | 명함 | −3.0° | 이름, 직급, 이메일, 휴대폰, 유선 |
| `plate_kr_new/old, plate_us.jpg` | 번호판 3종 | −2.0° | 번호판 문자, 규격 |

In [ ]:
def gen_business_registration(path="business_registration.jpg"):
    W, H = 620, 800
    img = Image.new("RGB", (W, H), (250, 250, 250))
    d = ImageDraw.Draw(img)

    d.rectangle([(20, 20), (W - 20, H - 20)], outline=(0, 0, 0), width=3)
    d.rectangle([(30, 30), (W - 30, H - 30)], outline=(120, 120, 120), width=1)
    d.text((190, 60), "사업자등록증", fill=(0, 0, 0), font=F(28, True))
    d.line([(50, 110), (W - 50, 110)], fill=(0, 0, 0), width=2)

    items = [
        ("등록번호",       "123-45-67890"),      # 정규식 매칭 대상
        ("법인명(상호)",   "(주)파이썬 인텔리전스"),
        ("대표자명",       "홍길동"),
        ("개업연월일",     "2024년 01월 01일"),
        ("사업장 소재지",  "서울특별시 강남구 테헤란로 123, 4층"),
        ("대표 전화번호",  "02-1234-5678"),      # 정규식 매칭 대상
        ("사업의 종류",    "업태: 정보통신업 / 종목: 소프트웨어 개발"),
    ]
    for i, (label, val) in enumerate(items):
        y = 150 + i * 55
        d.text((60, y),  f"- {label}", fill=(0, 0, 0), font=F(16, True))
        d.text((225, y), f":  {val}",  fill=(0, 0, 0), font=F(16))
        d.line([(50, y + 35), (W - 50, y + 35)], fill=(220, 220, 220), width=1)

    d.text((170, H - 120), "국 세 청 장  [직인생략]", fill=(0, 0, 0), font=F(26, True))
    img.save(path, "JPEG", quality=95)
    return path


def gen_receipt(path="receipt_sample.jpg", skew=-2.5):
    img = Image.new("RGB", (460, 620), (255, 255, 255))
    d = ImageDraw.Draw(img)
    rows = [
        ("[ AI 스마트 스토어 ]",           True,  120),
        ("-" * 34,                          False, 30),
        ("사업자번호: 214-88-12345",        False, 30),
        ("전화번호: 02-555-7890",           False, 30),
        ("주소: 서울시 강남구 테헤란로 456", False, 30),
        ("-" * 34,                          False, 30),
        ("상품명        수량        금액",   False, 30),
        ("-" * 34,                          False, 30),
        ("딥러닝 교재      1      35,000",  False, 30),
        ("아메리카노      2       9,000",   False, 30),
        ("-" * 34,                          False, 30),
        ("합계금액              44,000",    True,  30),   # 원본 정규식이 놓치던 표기
        ("=" * 34,                          False, 30),
        ("거래일자: 2026-08-27",            False, 30),
        ("이용해 주셔서 감사합니다.",        False, 100),
    ]
    y = 30
    for text, bold, x in rows:
        d.text((x, y), text, fill=(0, 0, 0), font=F(18 if bold else 15, bold))
        y += 36
    cv2.imwrite(path, rotate_pil(img, skew))
    return path


def gen_business_card(path="business_card_sample.jpg", skew=-3.0):
    img = Image.new("RGB", (620, 360), (255, 255, 255))
    d = ImageDraw.Draw(img)
    d.text((40, 36),  "(주)테크솔루션",        fill=(0, 51, 102), font=F(24, True))
    d.text((40, 88),  "홍길동 팀장",           fill=(0, 0, 0),    font=F(22, True))
    d.text((40, 124), "AI 개발팀 / 수석연구원", fill=(90, 90, 90), font=F(15))
    d.line([(40, 162), (580, 162)], fill=(200, 200, 200), width=2)
    for i, line in enumerate([
        "TEL : 02-1234-5678",
        "MOBILE : 010-9876-5432",
        "EMAIL : gd.hong@techsolution.co.kr",
        "ADDR : 서울특별시 강남구 테헤란로 123 5층",
    ]):
        d.text((40, 186 + i * 32), line, fill=(40, 40, 40), font=F(15))
    cv2.imwrite(path, rotate_pil(img, skew))
    return path


def gen_plates(skew=-2.0):
    paths = []

    # 한국 신형: 숫자3 + 한글1 + 숫자4
    p = Image.new("RGB", (520, 110), (255, 255, 255)); d = ImageDraw.Draw(p)
    d.rectangle([(0, 0), (40, 110)], fill=(0, 102, 204))          # 좌측 홀로그램 띠
    d.rectangle([(2, 2), (517, 107)], outline=(0, 0, 0), width=4)
    d.text((78, 26), "123가 4567", fill=(0, 0, 0), font=F(44, True))
    cv2.imwrite("plate_kr_new.jpg", rotate_pil(p, skew)); paths.append("plate_kr_new.jpg")

    # 한국 구형: 지역명 + 숫자2 + 한글1 + 숫자4
    p = Image.new("RGB", (520, 110), (245, 245, 220)); d = ImageDraw.Draw(p)
    d.rectangle([(2, 2), (517, 107)], outline=(0, 100, 0), width=4)
    d.text((52, 26), "서울 12가 3456", fill=(0, 100, 0), font=F(42, True))
    cv2.imwrite("plate_kr_old.jpg", rotate_pil(p, skew)); paths.append("plate_kr_old.jpg")

    # 미국 규격
    p = Image.new("RGB", (520, 260), (250, 250, 250)); d = ImageDraw.Draw(p)
    d.rectangle([(3, 3), (516, 256)], outline=(200, 0, 0), width=5)
    d.text((165, 28),  "CALIFORNIA", fill=(200, 0, 0),  font=F(22, True))
    d.text((112, 100), "7XYZ890",    fill=(0, 51, 102), font=F(56, True))
    cv2.imwrite("plate_us.jpg", rotate_pil(p, skew)); paths.append("plate_us.jpg")

    return paths


SAMPLES = [gen_business_registration(), gen_receipt(), gen_business_card()] + gen_plates()
for f in SAMPLES:
    print(f"{f:30} {cv2.imread(f).shape}")

### 눈으로 확인

**한글이 `???`나 네모로 보이면 폰트 문제다.** 2번 셀로 돌아가야 한다. 여기서 안 잡고 넘어가면 OCR이 안 되는 이유를 엉뚱한 데서 찾게 된다.

In [ ]:
show(cv2.imread("business_registration.jpg"),
     cv2.imread("receipt_sample.jpg"),
     cv2.imread("business_card_sample.jpg"),
     titles=["사업자등록증", "영수증 (-2.5° 기울임)", "명함 (-3.0° 기울임)"],
     height=6)

show(cv2.imread("plate_kr_new.jpg"),
     cv2.imread("plate_kr_old.jpg"),
     cv2.imread("plate_us.jpg"),
     titles=["한국 신형", "한국 구형", "미국"], height=3)

---
# 5. 실험 ①  —  Deskew 코드가 조용히 실패하는 순간

인터넷과 강의 자료에 널리 퍼진 deskew 코드는 전경(글자)을 이렇게 찾는다.

```python
coords = np.column_stack(np.where(gray < 255))   # 255가 아니면 전부 글자로 간주
```

합성 이미지는 배경이 정확히 255라 잘 돌아간다. 그런데 **실제 사진**은

- JPEG 압축 아티팩트로 254, 253이 섞이고
- 촬영 노이즈로 픽셀이 흔들리고
- 종이가 완전 백색이 아니라 250쯤 된다

이러면 **거의 모든 픽셀이 `< 255`가 되어** `minAreaRect`가 이미지 전체를 감싸버리고, 각도는 0도로 나온다.

**가장 나쁜 건 에러가 안 난다는 점이다.** 함수는 정상 리턴하고, 기울어진 이미지가 그대로 다음 단계로 넘어가고, 인식률만 조용히 나빠진다.

아래 셀에서 직접 확인해보자.

In [ ]:
def measure_angle(gray, mode="otsu"):
    """전경 검출 방식을 바꿔가며 보정각을 계산한다.
    mode: "otsu" | 정수 문자열(고정 임계값)
    반환값은 '이만큼 회전시켜 펴라'는 각도다."""
    if mode == "otsu":
        mask = cv2.threshold(gray, 0, 255,
                             cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
        coords = np.column_stack(np.where(mask > 0))
    else:
        coords = np.column_stack(np.where(gray < int(mode)))

    if coords.shape[0] == 0:
        return 0.0, 0.0

    ratio = coords.shape[0] / gray.size
    angle = cv2.minAreaRect(coords)[-1]
    angle = -(90 + angle) if angle < -45 else -angle
    return angle, ratio


def make_test_image(skew_deg, bg=255, noise=0, jpeg=False, seed=0):
    """텍스트 줄 3개를 그린 뒤 기울이고, 실사진 조건을 흉내낸다."""
    rng = np.random.default_rng(seed)
    img = np.full((300, 600), bg, np.uint8)
    for y in (80, 130, 180):
        cv2.rectangle(img, (60, y), (540, y + 16), 0, -1)
    M = cv2.getRotationMatrix2D((300, 150), skew_deg, 1.0)
    img = cv2.warpAffine(img, M, (600, 300), borderValue=bg)
    if noise:
        img = np.clip(img.astype(np.int16) + rng.normal(0, noise, img.shape),
                      0, 255).astype(np.uint8)
    if jpeg:
        img = cv2.imdecode(cv2.imencode(".jpg", img,
                                        [cv2.IMWRITE_JPEG_QUALITY, 85])[1], 0)
    return img


TRUE_SKEW = -5.0                # 실제로 -5도 기울였으니 보정각은 +5.00이 정답
CASES = [
    ("깨끗한 합성 이미지",       dict()),
    ("JPEG 압축 (q=85)",        dict(jpeg=True)),
    ("배경이 살짝 회색 (250)",   dict(bg=250)),
    ("촬영 노이즈 (sigma=6)",    dict(noise=6)),
    ("노이즈 + JPEG",           dict(noise=6, jpeg=True)),
    ("노이즈 (sigma=15)",       dict(noise=15, jpeg=True)),
]

print(f"{'상황':<26}{'gray<255':>18}{'gray<230':>12}{'Otsu':>10}")
print(f"{'':26}{'(각도 / 대상비율)':>18}")
print("-" * 68)
for name, kw in CASES:
    g = make_test_image(TRUE_SKEW, **kw)
    a255, r255 = measure_angle(g, "255")
    a230, _    = measure_angle(g, "230")
    aots, _    = measure_angle(g, "otsu")
    flag = "" if abs(aots - 5) < 1 else "  <- Otsu도 흔들림"
    print(f"{name:<26}{a255:>8.2f} /{r255*100:>5.0f}%{a230:>12.2f}{aots:>10.2f}{flag}")

print("-" * 68)
print("정답: 5.00   |   대상비율이 50%를 넘으면 gray<255 방식이 0도로 죽는다")

### 결론

**Otsu로 전경을 먼저 확정한다.** 이미지마다 임계값을 자동으로 정하기 때문에 배경 밝기가 250이든 200이든 글자와 배경 사이를 알아서 찾는다.

```
픽셀 수
  │        ╱▔▔╲                    ╱▔▔▔╲
  │       ╱     ╲                 ╱      ╲
  │      ╱  글자  ╲               ╱  배경   ╲
  │─────╱─────────╲─────────────╱──────────╲──▶ 밝기
  0                  ↑                        255
                  Otsu가 찾는 지점
```

---
# 6. 전처리 체인

```
[원본 BGR]
    ↓ ① Deskew (Otsu 기반)   기울기 보정
    ↓ ② Grayscale            3채널 → 1채널
    ↓ ③ Sharpening           글자 경계 강조
    ↓ ④ Gaussian Blur        샤프닝이 만든 노이즈 완화
    ↓ ⑤ Adaptive Threshold   조명 불균형에 강한 이진화
[이진 이미지]
```

**③ 샤프닝 다음에 ④ 블러가 오는 게 이상해 보이지만 의도가 있다.** 샤프닝은 경계를 강조하면서 노이즈도 같이 증폭시킨다. 약한 블러로 그 노이즈만 누르고 강조된 경계는 남긴다. 순서를 바꾸면(블러 → 샤프닝) 이미 뭉갠 경계를 다시 살리는 꼴이라 효과가 떨어진다.

In [ ]:
SHARPEN_KERNEL = np.array([[0, -1, 0],
                           [-1, 5, -1],
                           [0, -1, 0]])


def deskew(image, min_angle=0.5, verbose=False):
    """Otsu로 전경을 확정한 뒤 기울기를 보정한다."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image

    mask = cv2.threshold(gray, 0, 255,
                         cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    coords = np.column_stack(np.where(mask > 0))
    if coords.shape[0] == 0:
        return image

    angle = cv2.minAreaRect(coords)[-1]
    angle = -(90 + angle) if angle < -45 else -angle

    if verbose:
        print(f"  [deskew] 보정각 {angle:+.2f}도"
              f"{' (기준 미만이라 생략)' if abs(angle) < min_angle else ''}")

    if abs(angle) < min_angle:
        return image

    h, w = image.shape[:2]
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(image, M, (w, h),
                          flags=cv2.INTER_CUBIC,
                          borderMode=cv2.BORDER_CONSTANT,
                          borderValue=(255, 255, 255))


def preprocess(image_path, block_size=15, C=3, verbose=False):
    """전처리 체인 전체. (기울기 보정본, 이진화본)을 돌려준다."""
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"이미지를 로드할 수 없다: {image_path}")

    deskewed = deskew(img, verbose=verbose)                       # ①
    gray = cv2.cvtColor(deskewed, cv2.COLOR_BGR2GRAY)             # ②
    sharpened = cv2.filter2D(gray, -1, SHARPEN_KERNEL)            # ③
    blurred = cv2.GaussianBlur(sharpened, (3, 3), 0)              # ④
    binary = cv2.adaptiveThreshold(                               # ⑤
        blurred, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        block_size,   # 홀수여야 한다
        C,
    )
    return deskewed, binary


print("전처리 함수 준비 완료")

### 단계별로 눈으로 보기

**전처리는 에러 없이 조용히 실패하기 때문에 반드시 눈으로 확인해야 한다.** 디버그 저장 두 줄이 며칠을 아껴준다.

In [ ]:
raw = cv2.imread("receipt_sample.jpg")
print("영수증 (-2.5도로 기울여둔 이미지)")
dsk = deskew(raw, verbose=True)

gray = cv2.cvtColor(dsk, cv2.COLOR_BGR2GRAY)
shp  = cv2.filter2D(gray, -1, SHARPEN_KERNEL)
blr  = cv2.GaussianBlur(shp, (3, 3), 0)
bin_ = cv2.adaptiveThreshold(blr, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                             cv2.THRESH_BINARY, 15, 3)

show(raw, dsk, gray, shp, blr, bin_,
     titles=["① 원본(기울어짐)", "② Deskew", "③ Grayscale",
             "④ Sharpen", "⑤ Blur", "⑥ Adaptive Threshold"],
     cols=3, height=5)

### `blockSize`와 `C` 바꿔보기

기본값이 모든 이미지에 맞을 리가 없다. **어두운 실내에서 찍은 영수증은 반드시 조정해야 한다.**

| 파라미터 | 크게 하면 | 작게 하면 |
|---|---|---|
| `blockSize` | 넓은 영역 평균 → 전역 이진화에 가까워짐 | 국소 대비 강조, 노이즈에 민감 |
| `C` | 더 많이 흰색 → 글자가 얇아짐 | 더 많이 검은색 → 글자가 두꺼워지고 배경 노이즈 증가 |

In [ ]:
variants, titles = [], []
for bs, c in [(7, 2), (15, 3), (31, 5), (51, 10)]:
    variants.append(cv2.adaptiveThreshold(blr, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                          cv2.THRESH_BINARY, bs, c))
    titles.append(f"blockSize={bs}, C={c}")

show(*variants, titles=titles, cols=4, height=5)

---
# 7. 실험 ②  —  원본 정규식이 아무것도 못 잡는다

강의 자료의 패턴을 그대로 돌려본다. **두 개가 항상 `None`을 뱉는다.**

In [ ]:
import re

print("=" * 66)
print("  원본 금액 패턴:  (?:합계|결제금액|총액|AMOUNT)[\\s:]*([0-9,]+)")
print("=" * 66)
ORIG_AMOUNT = r'(?:합계|결제금액|총액|AMOUNT)[\s:]*([0-9,]+)'
for t in ["합계금액              44,000",
          "합계 금액:            40,000",
          "합계금액             6,500원",
          "총액: 12,000"]:
    m = re.search(ORIG_AMOUNT, t)
    print(f"  {t!r:34} -> {m.group(1) if m else 'None  <- 실패'}")

print()
print("  이유: '합계'가 매칭된 뒤 [\\s:]* 는 공백과 콜론만 먹는다.")
print("        실제로는 '금액' 두 글자가 끼어 있어서 [0-9,]+ 로 못 넘어간다.")
print("        |  교대는 왼쪽부터 시도하므로 짧은 '합계'가 먼저 이겨버린다.")

print()
print("=" * 66)
print("  원본 날짜 패턴:  \\b\\d{4}-./-./\\b")
print("=" * 66)
ORIG_DATE = r'\b\d{4}-./-./\b'
for t in ["거래일자: 2026-08-27", "2026-08-27 14:30:00", "구매일시: 2026-08-27 15:30"]:
    m = re.search(ORIG_DATE, t)
    print(f"  {t!r:34} -> {m.group(0) if m else 'None  <- 실패'}")

print()
print("  이유: '.'은 아무 문자 1개, '/'는 리터럴 슬래시다.")
print("        '2026-X/-Y/' 라는 존재하지 않는 형태를 찾고 있다.")

### 그리고 `\b`는 한글 문서에서 위험하다

파이썬 정규식의 `\b`는 `\w`와 `\W`의 경계인데, **`\w`에 한글이 포함된다.**

OCR은 공백을 자주 놓친다. "사업자번호: 123-45-67890"이 "사업자번호123-45-67890"으로 붙어 나오는 건 흔한 일이고, 그 순간 `\b` 때문에 매칭이 죽는다.

In [ ]:
print("한글이 \\w에 포함되나:", bool(re.match(r'\w', '호')))
print()

WITH_B    = r'\b\d{3}-\d{2}-\d{5}\b'
WITHOUT_B = r'\d{3}\s*-\s*\d{2}\s*-\s*\d{5}'

H1, H2 = "\\b 있음", "\\b 없음"          # f-string 안에 역슬래시를 못 넣어서 밖으로 뺐다
print(f"{'입력':<34}{H1:>10}{H2:>10}")
print("-" * 56)
for t in ["사업자번호: 123-45-67890",
          "사업자번호123-45-67890",
          "번호123-45-67890 확인"]:
    a = "매칭" if re.search(WITH_B, t) else "실패"
    b = "매칭" if re.search(WITHOUT_B, t) else "실패"
    print(f"{t:<34}{a:>10}{b:>10}")

### 번호판 문자 클래스도 깨져 있다

```python
r'\b\d{3}\s*[가-나|다-라|마|거-너|더-러|머-버|...]\s*\d{4}\b'
```

문제가 둘이다.

1. **`[...]` 안의 `|`는 교대 연산자가 아니라 그냥 파이프 문자다** → `123|4567`이 번호판으로 통과한다
2. **`가-나`는 유니코드 범위라 1177자를 포함한다** → `123뭐 4567` 같은 게 통과하고, 정작 렌터카 `123하 4567`은 못 잡는다

**엉뚱한 건 통과시키고 진짜는 놓친다.** 한국 번호판 용도 문자는 40개뿐이니 나열하는 게 맞다.

In [ ]:
ORIG_PLATE = (r'\b\d{3}\s*[가-나|다-라|마|거-너|더-러|머-버|서-어|저-처|'
              r'고-노|도-로|모-보|소-오|조-초|주|임]\s*\d{4}\b')

print("'가'~'나' 유니코드 범위에 든 글자 수:", ord('나') - ord('가') + 1)
print()
print(f"{'입력':<18}{'원본 패턴':>12}")
print("-" * 32)
for t in ["123가 4567", "123나 4567", "123뭐 4567", "123|4567", "123하 4567"]:
    print(f"{t:<18}{'매칭' if re.search(ORIG_PLATE, t) else '실패':>12}")
print()
print("  '뭐' 통과 / '|' 통과 / '하'(렌터카) 실패  ← 전부 잘못된 동작")

---
# 8. 고친 정규식 패턴

원칙 네 가지를 지켰다.

1. **긴 패턴을 `|` 왼쪽에** — `합계금액`이 `합계`보다 먼저
2. **`\b` 대신 `\s*`** — 한글 경계 문제 회피 + OCR 공백 변형 흡수
3. **유니코드 범위 대신 실제 문자 나열**
4. **`[0-9][0-9,]*`** — 쉼표로 시작하는 매칭 방지

In [ ]:
# 한국 번호판 용도 문자 40자 (자가용 + 영업용 + 택배 + 렌터카)
PLATE_CHARS = "가나다라마거너더러머버서어저고노도로모보소오조구누두루무부수우주바사아자배하허호"
REGIONS = ("서울|부산|대구|인천|광주|대전|울산|세종|경기|강원|"
           "충북|충남|전북|전남|경북|경남|제주")

PATTERNS = {
    # ── 공통 ──
    "business_number": r'\d{3}\s*-\s*\d{2}\s*-\s*\d{5}',
    "mobile":          r'01[016789]\s*-\s*\d{3,4}\s*-\s*\d{4}',
    "tel":             r'0\d{1,2}\s*-\s*\d{3,4}\s*-\s*\d{4}',
    "date":            r'\d{4}\s*[-/.]\s*\d{1,2}\s*[-/.]\s*\d{1,2}',
    "email":           r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',

    # ── 영수증: 긴 키워드를 왼쪽에! ──
    "amount": (r'(?:합\s*계\s*금\s*액|결\s*제\s*금\s*액|총\s*결\s*제|'
               r'합\s*계|총\s*액|TOTAL|AMOUNT)[\s:]*([0-9][0-9,]*)'),

    # ── 명함: 긴 직급을 왼쪽에! ──
    "name_title": (r'([가-힣]{2,4})\s*'
                   r'(수석연구원|책임연구원|선임연구원|연구원|'
                   r'대표이사|대표|이사|부장|차장|과장|대리|팀장|사원)'),

    # ── 번호판 ──
    "plate_kr_new": rf'\d{{3}}\s*[{PLATE_CHARS}]\s*\d{{4}}',
    "plate_kr_old": rf'(?:{REGIONS})\s*\d{{2}}\s*[{PLATE_CHARS}]\s*\d{{4}}',
    "plate_us":     r'\b[0-9A-Z][A-Z]{3}\s*[0-9]{3,4}\b',
}

print(f"패턴 {len(PATTERNS)}개 정의 완료 | 번호판 용도 문자 {len(PLATE_CHARS)}자")

### 검증 — 전부 통과하는지 확인

**패턴은 반드시 실제 OCR 출력 문자열에 돌려서 검증한다.** 정규식은 에러를 안 내고 그냥 못 찾을 뿐이라, 검증을 건너뛰면 모델을 의심하며 시간을 낭비하게 된다.

In [ ]:
TESTS = [
    # (키, 입력, 기대값)   기대값 None이면 "매칭되면 안 됨"
    ("business_number", "사업자번호: 123-45-67890",          "123-45-67890"),
    ("business_number", "등록번호 :  214-88-12345",          "214-88-12345"),
    ("amount",          "합계금액              44,000",      "44,000"),
    ("amount",          "합계 금액:            40,000",      "40,000"),
    ("amount",          "합계금액             6,500원",      "6,500"),
    ("amount",          "TOTAL:  12,000",                    "12,000"),
    ("date",            "거래일자: 2026-08-27",              "2026-08-27"),
    ("date",            "2026/8/27",                         "2026/8/27"),
    ("mobile",          "MOBILE : 010-9876-5432",            "010-9876-5432"),
    ("tel",             "TEL : 02-1234-5678",                "02-1234-5678"),
    ("email",           "EMAIL : gd.hong@techsolution.co.kr","gd.hong@techsolution.co.kr"),
    ("plate_kr_new",    "123가 4567",                        "123가 4567"),
    ("plate_kr_new",    "123가4567",                         "123가4567"),
    ("plate_kr_new",    "123하 4567",                        "123하 4567"),
    ("plate_kr_new",    "123뭐 4567",                        None),
    ("plate_kr_new",    "123|4567",                          None),
    ("plate_kr_old",    "서울 12가 3456",                    "서울 12가 3456"),
    ("plate_kr_old",    "123가 4567",                        None),
    ("plate_us",        "CALIFORNIA 7XYZ890",                "7XYZ890"),
]

passed = 0
print(f"{'키':16}{'입력':32}{'결과':28}판정")
print("-" * 84)
for key, text, expect in TESTS:
    m = re.search(PATTERNS[key], text, re.IGNORECASE)
    got = (m.group(1) if m.groups() else m.group(0)) if m else None
    ok = (got == expect)
    passed += ok
    print(f"{key:16}{text:32}{str(got):28}{'통과' if ok else '실패 <-'}")

print("-" * 84)
print(f"{passed}/{len(TESTS)} 통과")

---
# 9. Tesseract 실습

가장 오래된 오픈소스 엔진이다. **속도가 매우 빠르고 가볍지만**, 텍스트가 조금만 회전돼 있거나 배경이 복잡하면 인식률이 급격히 떨어진다.

Tesseract는 **이진화된 깨끗한 입력을 전제로 만들어진 엔진**이라 전처리 효과가 크다. 뒤에 나올 딥러닝 엔진과 정반대 성질이다.

In [ ]:
if not HAS_TESSERACT:
    print("Tesseract를 못 찾았다. 이 셀은 건너뛰고 EasyOCR로 진행하면 된다.")
else:
    import pytesseract
    from PIL import Image as PILImage

    LANG = "kor+eng" if "kor" in pytesseract.get_languages() else "eng"
    print(f"사용 언어: {LANG}\n")

    # 원본 vs 전처리본 비교 — Tesseract는 전처리 효과가 크다
    _, binary_img = preprocess("receipt_sample.jpg")

    for label, src in [("원본", PILImage.open("receipt_sample.jpg")),
                       ("전처리(deskew+이진화)", PILImage.fromarray(binary_img))]:
        text = pytesseract.image_to_string(src, lang=LANG)
        lines = [l for l in text.splitlines() if l.strip()]
        print("=" * 52)
        print(f"  {label}  —  {len(lines)}줄 인식")
        print("=" * 52)
        for l in lines:
            print("   ", l)
        print()

### 신뢰도까지 받고 싶으면 `image_to_data`

`image_to_string`은 문자열만 준다. **단어별 신뢰도가 필요하면 `image_to_data`를 쓴다** — 하이브리드 구조를 짜거나 결과를 필터링할 때 이게 결정적이다.

In [ ]:
if HAS_TESSERACT:
    from pytesseract import Output
    from PIL import Image as PILImage

    data = pytesseract.image_to_data(PILImage.open("receipt_sample.jpg"),
                                     lang=LANG, output_type=Output.DICT)
    print(f"{'텍스트':24}{'신뢰도':>8}")
    print("-" * 34)
    for txt, conf in zip(data["text"], data["conf"]):
        if txt.strip() and float(conf) > 0:
            print(f"{txt:24}{float(conf):>8.1f}")
else:
    print("Tesseract 없음 — 건너뜀")

---
# 10. EasyOCR 실습

PyTorch 기반이고 내부적으로 **CRAFT(검출) + CRNN(인식)** 딥러닝 모델을 쓴다. Tesseract 대비 휘어진 글자와 복잡한 배경에서 훨씬 강하고, **단어별 신뢰도를 기본으로 준다.**

> **첫 실행은 모델 가중치를 내려받느라 몇 분 걸린다.** 이후로는 캐시를 쓴다.

> `easyocr.Reader()` 생성은 **모델을 메모리에 올리는 비싼 작업**이다. 절대 루프나 함수 안에서 반복 생성하면 안 된다. 아래처럼 전역에 한 번만 만든다.

In [ ]:
READER = None

if HAS_EASYOCR:
    import easyocr
    print("EasyOCR 모델 로딩 중... (첫 실행은 다운로드 때문에 몇 분 걸린다)")
    READER = easyocr.Reader(["ko", "en"], gpu=False)   # GPU 있으면 gpu=True
    print("로딩 완료")
else:
    print("easyocr가 없다.  uv add easyocr  또는  pip install easyocr")

In [ ]:
def run_ocr(image):
    """EasyOCR 실행. 반환: [(bbox, text, confidence), ...]"""
    if READER is None:
        raise RuntimeError("EasyOCR이 준비되지 않았다")
    return READER.readtext(image)


def print_ocr(results, title="OCR 결과"):
    print("=" * 52)
    print(f"  {title}  —  {len(results)}건")
    print("=" * 52)
    for _, text, prob in results:
        bar = "#" * int(prob * 20)
        print(f"  {prob:.3f} {bar:<20} {text}")
    if results:
        confs = [p for _, _, p in results]
        print(f"\n  평균 신뢰도 {np.mean(confs):.3f} | 최저 {min(confs):.3f}")


if READER:
    res = run_ocr(cv2.imread("receipt_sample.jpg"))
    print_ocr(res, "영수증 (원본 입력)")

---
# 11. 실험 ③  —  이진화가 EasyOCR에 도움이 되나?

강의 자료의 파이프라인은 **이진화한 이미지를 EasyOCR에 넣는다.** 전통적인 OCR 교과서대로다.

그런데 **CRAFT는 자연 이미지로 학습된 모델**이다. 이진화로 정보를 날린 입력이 오히려 검출을 방해할 수 있다.

- **Tesseract** — 이진화가 도움된다. 애초에 이진 입력을 전제로 만들어졌다
- **EasyOCR / PaddleOCR** — 원본이나 그레이스케일이 더 나은 경우가 많다

**믿지 말고 측정한다.** "전처리는 무조건 좋다"는 Tesseract 시절의 감각이다.

In [ ]:
if READER:
    raw = cv2.imread("receipt_sample.jpg")
    dsk, binary_img = preprocess("receipt_sample.jpg")
    gray_img = cv2.cvtColor(dsk, cv2.COLOR_BGR2GRAY)

    print(f"{'입력 형태':<24}{'검출 건수':>10}{'평균 신뢰도':>14}{'최저':>10}")
    print("-" * 60)
    ab_results = {}
    for name, img in [("① 원본 (기울어짐)", raw),
                      ("② Deskew만",        dsk),
                      ("③ Deskew+그레이",   gray_img),
                      ("④ Deskew+이진화",   binary_img)]:
        r = run_ocr(img)
        ab_results[name] = r
        if r:
            confs = [p for _, _, p in r]
            print(f"{name:<24}{len(r):>10}{np.mean(confs):>14.3f}{min(confs):>10.3f}")
        else:
            print(f"{name:<24}{0:>10}{'-':>14}{'-':>10}")

    print("-" * 60)
    print("검출 건수와 평균 신뢰도를 같이 봐야 한다.")
    print("건수가 많아도 신뢰도가 낮으면 쪼개져서 잘못 읽힌 것일 수 있다.")
else:
    print("EasyOCR 없음 — 건너뜀")

In [ ]:
if READER:
    # 가장 좋았던 입력의 실제 텍스트를 확인한다
    best = max(ab_results.items(),
               key=lambda kv: np.mean([p for _, _, p in kv[1]]) if kv[1] else 0)
    print_ocr(best[1], f"가장 신뢰도가 높았던 입력: {best[0]}")

### 실제로 돌려본 결과

이 노트북을 그대로 실행했을 때 나온 수치다.

```
입력 형태                검출 건수    평균 신뢰도      최저
------------------------------------------------------
① 원본 (기울어짐)            15       0.880      0.447
② Deskew만                 15       0.869      0.266
③ Deskew+그레이             15       0.869      0.266
④ Deskew+이진화             15       0.574      0.069   <- 크게 나빠짐
```

**이진화가 평균 신뢰도를 0.880에서 0.574로 떨어뜨렸다.** 검출 건수는 15건으로 같은데 확신도만 무너진 것이다.

원본 강의 코드는 `process()`에서 **이진화 이미지를 EasyOCR에 넘긴다.** 그래서 이 노트북의 `DocumentOCR.process()`는 `use_binary=False`가 기본값이다.

> 이 결과가 모든 이미지에 해당하는 건 아니다. 조명이 심하게 불균일한 실제 촬영본에서는 전처리가 이길 수도 있다. **중요한 건 "전처리는 좋다"고 믿지 말고 자기 데이터로 재보는 것이다.**

---
# 12. 통합 클래스 `DocumentOCR`

문서 종류가 달라도 구조는 같으니 하나로 묶는다. **`PATTERNS`에서 뽑을 키만 바꿔 끼우면** 사업자등록증이든 명함이든 번호판이든 그대로 돌아간다.

설계 원칙

- **단일 책임** — 각 메서드가 파이프라인의 한 단계만 맡는다
- **Reader 1회 생성** — `__init__`에서만 만든다
- **`use_binary` 스위치** — 실험 ③의 결과에 따라 입력을 고를 수 있게
- **실패 로그** — 파싱이 `None`일 때 왜 그런지 알 수 있게 `raw_text`를 남긴다

In [ ]:
from typing import Dict, Any, List, Tuple


class DocumentOCR:
    """전처리 → EasyOCR → 정규식 파싱을 묶은 문서 OCR 클래스"""

    def __init__(self, reader=None, languages=None, gpu=False):
        if reader is not None:
            self.reader = reader                     # 이미 만든 Reader 재사용
        else:
            import easyocr
            self.reader = easyocr.Reader(languages or ["ko", "en"], gpu=gpu)

    # ---------- 1) 전처리 ----------
    def preprocess(self, image_path, block_size=15, C=3, verbose=False):
        return preprocess(image_path, block_size, C, verbose)

    # ---------- 2) OCR ----------
    def execute_ocr(self, image):
        return self.reader.readtext(image)

    # ---------- 3) 파싱 ----------
    def parse(self, ocr_results, keys, conf_threshold=0.35, verbose=True):
        valid = [t.strip() for _, t, p in ocr_results if p >= conf_threshold]
        dropped = len(ocr_results) - len(valid)
        full_text = "\n".join(valid)

        data = {"raw_text": full_text}
        for key in keys:
            m = re.search(PATTERNS[key], full_text, re.IGNORECASE)
            data[key] = (m.group(1) if m.groups() else m.group(0)) if m else None

        if verbose:
            failed = [k for k in keys if data[k] is None]
            print(f"  [parse] 신뢰도 {conf_threshold} 이상 {len(valid)}건 채택, {dropped}건 폐기")
            if failed:
                print(f"  [parse] 파싱 실패: {failed}  <- raw_text를 확인할 것")
        return data

    # ---------- 4) 전체 실행 ----------
    def process(self, image_path, keys, conf_threshold=0.35,
                use_binary=False, verbose=True):
        if verbose:
            print(f"[{image_path}]")
        deskewed, binary = self.preprocess(image_path, verbose=verbose)
        target = binary if use_binary else deskewed
        results = self.execute_ocr(target)
        if verbose:
            print(f"  [ocr] {len(results)}건 검출")
        return self.parse(results, keys, conf_threshold, verbose)


def report(title, data, keys):
    """파싱 결과를 보기 좋게 출력한다."""
    print("\n" + "=" * 52)
    print(f"  [ {title} ]")
    print("=" * 52)
    for k in keys:
        v = data.get(k)
        mark = " " if v else "!"
        print(f" {mark} {k:18}: {v}")
    print("-" * 52)
    print("[Raw Text]")
    print(data.get("raw_text", ""))
    print("=" * 52)


ocr = DocumentOCR(reader=READER) if READER else None
print("DocumentOCR 준비 완료" if ocr else "EasyOCR이 없어 클래스만 정의됨")

---
# 13. 사업자등록증 OCR  (6.5)

뽑을 것: **사업자번호**, **전화번호**

In [ ]:
KEYS_BIZ = ["business_number", "tel"]

if ocr:
    result = ocr.process("business_registration.jpg",
                         keys=KEYS_BIZ, conf_threshold=0.40)
    report("사업자등록증", result, KEYS_BIZ)
else:
    print("EasyOCR 없음 — 건너뜀")

---
# 14. 영수증 OCR  (6.6)

뽑을 것: **사업자번호**, **전화번호**, **거래일자**, **결제금액**

원본 코드에서 `date`와 `amount`가 항상 `None`이던 바로 그 케이스다. 고친 패턴으로 잡히는지 확인한다.

In [ ]:
KEYS_RECEIPT = ["business_number", "tel", "date", "amount"]

if ocr:
    result = ocr.process("receipt_sample.jpg",
                         keys=KEYS_RECEIPT, conf_threshold=0.35)
    report("영수증", result, KEYS_RECEIPT)
else:
    print("EasyOCR 없음 — 건너뜀")

---
# 15. 명함 OCR  (6.7)

뽑을 것: **이름/직급**, **이메일**, **휴대폰**, **유선전화**

> 원본 6.7 절은 **이미지 생성 스크립트와 OCR 클래스가 한 코드 블록에 섞여 있어서** 그대로는 실행이 안 된다. 여기서는 분리했다.

### 휴대폰과 유선이 겹치는 문제

`tel` 패턴 `0\d{1,2}-\d{3,4}-\d{4}`는 **휴대폰번호도 매칭한다.** `010-9876-5432`도 `0` + `10` + `9876` + `5432`로 읽히기 때문이다.

**휴대폰을 먼저 뽑고, 그걸 제외한 나머지에서 유선을 찾는다.** 정규식 하나로 해결하려 하지 말고 파이썬 로직으로 나누는 게 훨씬 읽기 쉽다.

In [ ]:
def parse_business_card(ocr_results, conf_threshold=0.30, verbose=True):
    """명함 전용 파서 — 휴대폰/유선 중복을 로직으로 분리한다."""
    valid = [t.strip() for _, t, p in ocr_results if p >= conf_threshold]
    full_text = "\n".join(valid)

    data = {"name": None, "title": None, "email": None,
            "mobile": None, "tel": None, "raw_text": full_text}

    # 이름 + 직급 (캡처 그룹 2개)
    m = re.search(PATTERNS["name_title"], full_text)
    if m:
        data["name"], data["title"] = m.group(1), m.group(2)

    # 이메일
    m = re.search(PATTERNS["email"], full_text)
    if m:
        data["email"] = m.group(0)

    # 휴대폰 먼저
    m = re.search(PATTERNS["mobile"], full_text)
    if m:
        data["mobile"] = m.group(0)

    # 유선은 휴대폰과 다른 첫 번째 후보
    for cand in re.findall(PATTERNS["tel"], full_text):
        if cand != data["mobile"]:
            data["tel"] = cand
            break

    if verbose:
        failed = [k for k in ("name", "title", "email", "mobile", "tel")
                  if data[k] is None]
        print(f"  [parse] {len(valid)}건 채택" +
              (f" | 실패: {failed}" if failed else ""))
    return data


KEYS_CARD = ["name", "title", "email", "mobile", "tel"]

if ocr:
    print("[business_card_sample.jpg]")
    dsk, _ = ocr.preprocess("business_card_sample.jpg", verbose=True)
    res = ocr.execute_ocr(dsk)
    print(f"  [ocr] {len(res)}건 검출")
    result = parse_business_card(res, conf_threshold=0.30)
    report("명함", result, KEYS_CARD)
else:
    print("EasyOCR 없음 — 건너뜀")

---
# 16. 자동차 번호판 OCR  (6.8)

규격 3종(한국 신형 / 한국 구형 / 미국)을 순서대로 검사한다.

**패턴 순서가 중요하다.** 원본은 구형 패턴의 지역명이 선택(`?`)이라 신형과 충돌할 여지가 있었다. 여기서는 지역명을 필수로 만들어 겹치지 않게 했다.

In [ ]:
def parse_plate(ocr_results, conf_threshold=0.25, verbose=True):
    """번호판 규격별 정규식을 순차 적용한다."""
    valid = [t.strip() for _, t, p in ocr_results if p >= conf_threshold]
    full_text = " ".join(valid)

    ORDER = [("KR_OLD", "plate_kr_old"),      # 지역명 포함 → 더 구체적이라 먼저
             ("KR_NEW", "plate_kr_new"),
             ("US",     "plate_us")]

    for label, key in ORDER:
        m = re.search(PATTERNS[key], full_text, re.IGNORECASE)
        if m:
            return {"license_plate": m.group(0), "plate_type": label,
                    "raw_text": full_text}

    if verbose:
        print(f"  [parse] 어떤 규격에도 안 맞음 | raw={full_text!r}")
    return {"license_plate": None, "plate_type": "UNKNOWN", "raw_text": full_text}


if ocr:
    for title, path in [("한국 신형", "plate_kr_new.jpg"),
                        ("한국 구형", "plate_kr_old.jpg"),
                        ("미국 규격", "plate_us.jpg")]:
        try:
            print(f"\n[{path}]")
            dsk, _ = ocr.preprocess(path, verbose=True)
            res = ocr.execute_ocr(dsk)
            print(f"  [ocr] {len(res)}건 검출")
            r = parse_plate(res, conf_threshold=0.25)
            print(f"\n  {title}")
            print(f"    규격 분류 : {r['plate_type']}")
            print(f"    번호판    : {r['license_plate']}")
            print(f"    전체 인식 : {r['raw_text']}")
        except Exception as e:
            print(f"  [{title}] 처리 실패: {e}")
else:
    print("EasyOCR 없음 — 건너뜀")

---
# 17. 실험 ④  —  OCR이 "그럴듯하게" 틀리게 읽을 때

위 15·16번 셀을 실행하면 **두 개가 `None`으로 나온다.** 이건 정규식 버그가 아니라 OCR이 실제로 잘못 읽은 것이다.

```
명함 이메일   기대: gd.hong@techsolution.co.kr
              실제: gd hong@techsolutioncokr      <- 점(.)이 전부 사라짐

미국 번호판   기대: 7XYZ890
              실제: 7XYZ89O                       <- 숫자 0을 알파벳 O로 읽음
```

**OCR은 조용히 틀린다.** 에러도 안 나고 신뢰도도 높게 나온다. 사람이 보기 전엔 모른다.

## 글자 혼동 (Character Confusion)

모양이 비슷한 글자끼리 서로 오인식된다. OCR을 실무에 쓸 때 가장 자주 만나는 문제다.

| 숫자 | ↔ | 알파벳 | 비고 |
|---|---|---|---|
| `0` | ↔ | `O` `o` `D` `Q` | 가장 흔하다 |
| `1` | ↔ | `I` `l` `\|` | 산세리프 폰트에서 특히 |
| `5` | ↔ | `S` `s` | |
| `8` | ↔ | `B` | |
| `2` | ↔ | `Z` `z` | |
| `6` | ↔ | `G` `b` | |

**해결의 핵심은 "어느 자리가 숫자여야 하는지 안다"는 것이다.** 번호판 뒷 4자리는 반드시 숫자니까, 거기 나온 `O`는 무조건 `0`이다. 문맥을 알면 기계적으로 고칠 수 있다.

In [ ]:
# 숫자 자리에서 알파벳으로 잘못 읽힌 것을 되돌린다
DIGIT_FIX = str.maketrans({
    "O": "0", "o": "0", "D": "0", "Q": "0", "U": "0",
    "I": "1", "l": "1", "|": "1", "i": "1",
    "S": "5", "s": "5", "B": "8",
    "Z": "2", "z": "2", "G": "6", "b": "6",
    "T": "7", "A": "4", "g": "9", "q": "9",
})

# 반대 방향 (알파벳 자리에서 숫자로 잘못 읽힌 것)
ALPHA_FIX = str.maketrans({"0": "O", "1": "I", "5": "S", "8": "B", "2": "Z", "6": "G"})

# 숫자 자리에 나타날 수 있는 글자 모음 (관대한 패턴용)
DIGITISH = "0-9OoDQUIliSsBZzGbTAgq"

# 관대한 번호판 패턴 — 일단 잡고 나서 보정한다
LOOSE = {
    "plate_us":     rf'\b([0-9A-Z])([A-Z]{{3}})\s*([{DIGITISH}]{{3,4}})\b',
    "plate_kr_new": rf'([{DIGITISH}]{{3}})\s*([{PLATE_CHARS}])\s*([{DIGITISH}]{{4}})',
    "plate_kr_old": rf'({REGIONS})\s*([{DIGITISH}]{{2}})\s*'
                    rf'([{PLATE_CHARS}])\s*([{DIGITISH}]{{4}})',   # 지역명도 캡처
}


def parse_plate_robust(ocr_results, conf_threshold=0.25, verbose=True):
    """엄격한 패턴을 먼저 시도하고, 실패하면 관대한 패턴 + 글자 보정을 쓴다."""
    valid = [t.strip() for _, t, p in ocr_results if p >= conf_threshold]
    full_text = " ".join(valid)

    ORDER = [("KR_OLD", "plate_kr_old"), ("KR_NEW", "plate_kr_new"), ("US", "plate_us")]

    # 1차: 엄격한 패턴 (오인식이 없었다면 여기서 끝난다)
    for label, key in ORDER:
        m = re.search(PATTERNS[key], full_text, re.IGNORECASE)
        if m:
            return {"license_plate": m.group(0), "plate_type": label,
                    "corrected": False, "raw_text": full_text}

    # 2차: 관대한 패턴으로 잡고 숫자 자리만 보정
    for label, key in ORDER:
        m = re.search(LOOSE[key], full_text)
        if not m:
            continue
        g = m.groups()
        if key == "plate_us":
            plate = g[0] + g[1] + g[2].translate(DIGIT_FIX)
        elif key == "plate_kr_old":                     # 지역명 + 숫자2 + 한글 + 숫자4
            plate = g[0] + " " + g[1].translate(DIGIT_FIX) + g[2] + g[3].translate(DIGIT_FIX)
        else:                                            # 숫자3 + 한글 + 숫자4
            plate = g[0].translate(DIGIT_FIX) + g[1] + g[2].translate(DIGIT_FIX)
        if verbose:
            print(f"  [fix] 오인식 보정: {m.group(0)!r} -> {plate!r}")
        return {"license_plate": plate, "plate_type": label,
                "corrected": True, "raw_text": full_text}

    if verbose:
        print(f"  [parse] 어떤 규격에도 안 맞음 | raw={full_text!r}")
    return {"license_plate": None, "plate_type": "UNKNOWN",
            "corrected": False, "raw_text": full_text}


# 실제 OCR이 뱉었던 문자열로 검증
print("보정 로직 검증")
print("-" * 46)
for raw, want in [("CALIFORNIA 7XYZ89O", "7XYZ890"),
                  ("l23가 456O",          "123가4560"),
                  ("서울 I2가 3456",       "서울 12가 3456")]:
    fake = [(None, raw, 0.9)]
    r = parse_plate_robust(fake, verbose=False)
    got = r["license_plate"]
    print(f"  {raw:22} -> {str(got):16}{'OK' if got and got.replace(' ','')==want.replace(' ','') else ''}")

## 이메일에서 점이 사라지는 문제

`gd.hong@techsolution.co.kr` → `gd hong@techsolutioncokr`

마침표가 **공백으로 바뀌거나 아예 사라졌다.** 마침표는 픽셀 몇 개짜리라 OCR이 가장 자주 놓치는 문자다.

- **로컬 파트의 공백** → 마침표로 되돌리면 된다
- **도메인의 사라진 점** → `cokr`에서 `.co.kr`을 복원해야 한다. **알려진 TLD 목록**으로 뒤에서부터 맞춰본다

> 이건 **완벽하게 복원할 수 없다.** `abccom`이 `abc.com`인지 `abcc.om`인지는 원리적으로 모른다. 그래서 실무에서는 **회사 도메인 사전**을 두고 편집 거리로 맞추거나, 신뢰도가 낮은 항목은 사람 확인으로 넘긴다.

In [ ]:
# 관대한 이메일 패턴 — 로컬 파트에 공백을, 도메인에 점 없음을 허용한다
EMAIL_LOOSE = r'([A-Za-z0-9._%+\- ]{2,}?)\s*@\s*([A-Za-z0-9.\-]{3,})'

KNOWN_TLDS = ["co.kr", "or.kr", "ne.kr", "go.kr", "ac.kr", "re.kr",
              "com", "net", "org", "edu", "gov", "io", "ai", "kr"]


def repair_domain(domain):
    """점이 사라진 도메인을 알려진 TLD로 복원한다. 긴 TLD부터 시도."""
    if "." in domain:
        return domain
    for tld in sorted(KNOWN_TLDS, key=len, reverse=True):
        flat = tld.replace(".", "")
        if domain.lower().endswith(flat) and len(domain) > len(flat):
            return domain[:-len(flat)] + "." + tld
    return domain


def extract_email(full_text, verbose=True):
    """엄격한 패턴 우선, 실패하면 관대한 패턴 + 복원."""
    m = re.search(PATTERNS["email"], full_text)
    if m:
        return m.group(0), False

    m = re.search(EMAIL_LOOSE, full_text)
    if not m:
        return None, False

    local = m.group(1).strip().replace(" ", ".")     # 공백 -> 마침표
    domain = repair_domain(m.group(2))
    email = f"{local}@{domain}"
    if verbose:
        print(f"  [fix] 이메일 복원: {m.group(0).strip()!r} -> {email!r}")
    return email, True


print("도메인 복원 검증")
print("-" * 46)
for d, want in [("techsolutioncokr", "techsolution.co.kr"),
                ("navercom",         "naver.com"),
                ("gmailcom",         "gmail.com"),
                ("xyzkr",            "xyz.kr"),
                ("abc.co.kr",        "abc.co.kr")]:
    got = repair_domain(d)
    print(f"  {d:20} -> {got:22}{'OK' if got == want else 'X'}")

print()
print("이메일 전체 복원")
print("-" * 46)
e, fixed = extract_email("EMAIL : gd hong@techsolutioncokr")
print(f"  결과: {e}   (정답 gd.hong@techsolution.co.kr)")

## 보정을 적용해서 명함과 번호판 다시 돌리기

In [ ]:
def parse_business_card_robust(ocr_results, conf_threshold=0.30, verbose=True):
    """오인식 보정을 붙인 명함 파서."""
    valid = [t.strip() for _, t, p in ocr_results if p >= conf_threshold]
    full_text = "\n".join(valid)

    data = {"name": None, "title": None, "email": None,
            "mobile": None, "tel": None, "raw_text": full_text}

    m = re.search(PATTERNS["name_title"], full_text)
    if m:
        data["name"], data["title"] = m.group(1), m.group(2)

    data["email"], _ = extract_email(full_text, verbose=verbose)

    m = re.search(PATTERNS["mobile"], full_text)
    if m:
        data["mobile"] = m.group(0)

    for cand in re.findall(PATTERNS["tel"], full_text):
        if cand != data["mobile"]:
            data["tel"] = cand
            break

    if verbose:
        failed = [k for k in ("name", "title", "email", "mobile", "tel") if data[k] is None]
        print(f"  [parse] {len(valid)}건 채택" + (f" | 실패: {failed}" if failed else " | 전부 성공"))
    return data


if ocr:
    print("[명함 — 보정 적용]")
    dsk, _ = ocr.preprocess("business_card_sample.jpg", verbose=True)
    res = ocr.execute_ocr(dsk)
    print(f"  [ocr] {len(res)}건 검출")
    result = parse_business_card_robust(res, conf_threshold=0.30)
    report("명함 (보정 후)", result, KEYS_CARD)

    print("\n[번호판 — 보정 적용]")
    for title, path in [("한국 신형", "plate_kr_new.jpg"),
                        ("한국 구형", "plate_kr_old.jpg"),
                        ("미국 규격", "plate_us.jpg")]:
        dsk, _ = ocr.preprocess(path)
        r = parse_plate_robust(ocr.execute_ocr(dsk), conf_threshold=0.25)
        tag = " (보정됨)" if r["corrected"] else ""
        print(f"  {title:10} [{r['plate_type']:7}] {r['license_plate']}{tag}"
              f"   raw={r['raw_text']!r}")
else:
    print("EasyOCR 없음 — 건너뜀")

> ⚠️ **보정은 양날의 검이다.** `O → 0`을 무조건 적용하면 진짜 알파벳 `O`가 들어간 문자열이 망가진다.
> 그래서 위 코드는 **엄격한 패턴을 먼저 시도하고, 실패했을 때만** 관대한 패턴 + 보정으로 내려간다.
> 그리고 보정이 일어났다는 사실을 `corrected` 플래그와 로그로 남긴다 — **나중에 "이 값 왜 이래?"를 추적할 유일한 단서다.**

---
# 18. 신뢰도 임계값을 데이터로 정하기

원본 코드는 문서별로 다른 값을 쓴다 — 사업자등록증 0.40, 영수증 0.35, 명함 0.30, 번호판 0.25.

**감으로 정한 값을 그대로 쓰면 안 된다.** 너무 높이면 맞은 결과까지 버려서 필요한 정보가 `None`이 되고, 너무 낮추면 쓰레기가 섞여 정규식이 엉뚱한 걸 잡는다.

실제 이미지를 돌려서 **신뢰도 분포를 보고** 정한다.

In [ ]:
if ocr:
    print(f"{'문서':16}{'건수':>6}{'최저':>8}{'25%':>8}{'중앙':>8}{'평균':>8}")
    print("-" * 56)
    all_confs = {}
    for name, path in [("사업자등록증", "business_registration.jpg"),
                       ("영수증",       "receipt_sample.jpg"),
                       ("명함",         "business_card_sample.jpg"),
                       ("번호판(신형)", "plate_kr_new.jpg")]:
        dsk, _ = ocr.preprocess(path)
        r = ocr.execute_ocr(dsk)
        c = [p for _, _, p in r]
        all_confs[name] = c
        if c:
            print(f"{name:16}{len(c):>6}{min(c):>8.3f}"
                  f"{np.percentile(c,25):>8.3f}{np.median(c):>8.3f}{np.mean(c):>8.3f}")

    plt.figure(figsize=(8, 3.4))
    plt.boxplot(all_confs.values(), labels=range(1, len(all_confs) + 1), vert=False)
    plt.yticks(range(1, len(all_confs) + 1),
               [f"[{i}]" for i in range(1, len(all_confs) + 1)])
    plt.xlabel("confidence")
    plt.title("Confidence distribution by document type")
    plt.axvline(0.35, color="r", ls="--", lw=1, label="threshold 0.35")
    plt.legend(); plt.tight_layout(); plt.show()
    print("범례:", {i+1: n for i, n in enumerate(all_confs)})
else:
    print("EasyOCR 없음 — 건너뜀")

---
# 19. Naver Clova OCR API (참고)

키가 있어야 실행된다. 국내 최고 수준의 한국어 인식률을 가진 SaaS API다.

원본 코드에서 고친 부분

- **시크릿 키를 환경변수로** — 코드에 직접 쓰면 깃에 올라가는 순간 사고다
- **`with` 문으로 파일 핸들 관리** — 원본은 `open()`이 안 닫혀서 반복 호출 시 핸들이 샌다
- **`format`을 실제 확장자와 맞춤** — 원본은 `'png'`인데 `.jpg`를 보내고 있었다

In [ ]:
import os, json, uuid, time

def clova_ocr(image_path,
              api_url=None,
              secret_key=None,
              timeout=30):
    """Naver Clova OCR 호출. 환경변수 CLOVA_OCR_URL / CLOVA_OCR_SECRET 사용."""
    import requests

    api_url = api_url or os.environ.get("CLOVA_OCR_URL")
    secret_key = secret_key or os.environ.get("CLOVA_OCR_SECRET")
    if not api_url or not secret_key:
        raise RuntimeError(
            "환경변수가 없다.\n"
            "  export CLOVA_OCR_URL='https://...'\n"
            "  export CLOVA_OCR_SECRET='...'"
        )

    ext = os.path.splitext(image_path)[1].lstrip(".").lower()
    ext = "jpg" if ext in ("jpg", "jpeg") else ext      # 실제 확장자와 맞춘다

    request_json = {
        "images": [{"format": ext, "name": "demo"}],
        "requestId": str(uuid.uuid4()),
        "version": "V2",
        "timestamp": int(round(time.time() * 1000)),
    }
    payload = {"message": json.dumps(request_json).encode("UTF-8")}
    headers = {"X-OCR-SECRET": secret_key}

    with open(image_path, "rb") as f:                   # 핸들 누수 방지
        files = [("file", f)]
        resp = requests.post(api_url, headers=headers,
                             data=payload, files=files, timeout=timeout)
    resp.raise_for_status()
    return resp.json()


try:
    result = clova_ocr("receipt_sample.jpg")
    print("=== Naver Clova 인식 결과 ===")
    for image in result.get("images", []):
        for field in image.get("fields", []):
            print(f"  {field['inferConfidence']:.3f}  {field['inferText']}")
except Exception as e:
    print(f"실행 안 됨: {e}")

---
# 20. 정리

## 이 노트북에서 확인한 것

| 실험 | 발견 |
|---|---|
| **① Deskew** | `gray < 255`는 노이즈·JPEG·회색 배경에서 **에러 없이 0도를 반환**한다. Otsu로 바꿔야 한다 |
| **② 정규식** | 원본의 금액·날짜 패턴이 **항상 `None`**. `\|` 교대 순서와 문법 오류 때문 |
| **③ 이진화** | 딥러닝 엔진(EasyOCR)에는 이진화가 **도움이 안 될 수 있다**. 반드시 측정해서 정한다 |

## 실무 체크리스트

**전처리**

- [ ] `cv2.imread`가 `None`을 반환하는 경우를 처리했나 (경로 오타, 한글 경로)
- [ ] Deskew 전경 검출을 **Otsu**로 했나
- [ ] 보정각을 로그로 남겼나 — 0도만 계속 나오면 deskew가 죽은 것이다
- [ ] `blockSize`가 홀수인가 (짝수면 에러)
- [ ] 전처리 전후 이미지를 **실제로 눈으로 봤나**

**후처리**

- [ ] 정규식을 **실제 OCR 출력 문자열**에 돌려봤나 (샘플 문자열 말고)
- [ ] `|` 교대에서 **긴 패턴을 왼쪽에** 뒀나
- [ ] 한글 문서에서 `\b`를 피했나 (한글은 `\w`다)
- [ ] 문자 클래스 `[...]` 안에 `|`를 넣지 않았나
- [ ] 겹치는 패턴(휴대폰/유선)의 우선순위를 처리했나
- [ ] 임계값을 **분포를 보고** 정했나
- [ ] 파싱 실패 시 `raw_text`를 로그로 남기나

**운영**

- [ ] `easyocr.Reader`를 요청마다 만들고 있지 않나
- [ ] 신뢰도를 로그에 쌓고 있나 (OCR은 **조용히 틀린다**)
- [ ] 문서 종류를 먼저 분류하고 파서를 나눴나

## 더 해볼 것

1. **직접 찍은 사진을 넣어본다.** 합성 이미지에서 잘 되는 건 아무 증거가 못 된다 — 이번 deskew 함수가 정확히 그 사례였다
2. **이미지를 15도쯤 크게 기울여본다.** `warpAffine`이 원본 크기를 유지하므로 모서리가 잘린다. 캔버스를 키우는 코드가 필요해진다
3. **90도 돌아간 사진을 넣어본다.** `minAreaRect` 방식은 ±45도 이내만 잡는다. `pytesseract.image_to_osd()`가 필요하다
4. **하이브리드를 만들어본다.** 신뢰도가 임계값 미만인 건만 Clova로 재처리하는 구조

## 한 줄 정리

**OCR 코드는 에러 없이 조용히 실패한다.** 그래서 "돌아간다"가 아니라 **"제대로 되고 있다"를 측정해서 확인하는 습관**이 전부다.